In [ ]:
# Colab/bootstrap: clone this repository and install it editable.
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/JeonDongJun/mindscopex_analysis"
MARK_REL = Path("src") / "mindscopex_analysis" / "__init__.py"


def find_repo_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / MARK_REL).is_file():
            return path
    return None


root = find_repo_root()
if root is None:
    workdir = Path(os.environ.get("COLAB_REPO_DIR", "/content/mindscopex_analysis"))
    if (workdir / MARK_REL).is_file():
        subprocess.run(["git", "-C", str(workdir), "pull", "--ff-only"], check=False)
        root = workdir
    else:
        workdir.parent.mkdir(parents=True, exist_ok=True)
        if workdir.exists():
            shutil.rmtree(workdir)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(workdir)])
        root = workdir

os.environ["MINDSCOPEX_ROOT"] = str(root.resolve())
os.chdir(root)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
print("ready:", root)


# 12. Decoder Geometry

질문: top lure feature 후보들이 서로 비슷한 decoder direction을 갖는가?

강한 후보가 여러 개라면 단일 feature가 아니라 feature family일 수 있습니다. decoder cosine으로 후보 간 방향성을 확인합니다.

In [ ]:
from pathlib import Path
import os
import sys

root = Path(os.environ.get("MINDSCOPEX_ROOT", Path.cwd())).resolve()
if not (root / "src" / "mindscopex_analysis" / "__init__.py").is_file():
    for candidate in [root, *root.parents]:
        if (candidate / "src" / "mindscopex_analysis" / "__init__.py").is_file():
            root = candidate
            break
    else:
        raise RuntimeError("Could not find repository root. Run the clone cell first.")

src_path = str(root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(root)


In [ ]:
import torch
from IPython.display import display

from mindscopex_analysis import (
    BAT_BALL_CASE,
    DEFAULT_MODEL_ID,
    DEFAULT_QWEN_SCOPE_REPO_ID,
    LureCase,
    answer_logprob_margin,
    answer_variant_rows,
    bat_ball_answer_variants,
    bat_ball_paraphrases,
    candidate_feature_rows,
    case_transfer_rows,
    coefficient_sweep_for_handle,
    control_delta_bypass_rows,
    crt_transfer_cases,
    decoder_cosine_rows,
    default_sae_device,
    dtype_from_name,
    feature_handle_from_result,
    instruct_lure_case,
    intervention_mode_rows,
    layer_feature_search_rows,
    load_or_discover_handle_and_sae,
    load_qwen_language_model,
    load_qwen_scope_sae,
    prompt_token_window_rows,
    rank_lure_feature_effects,
    recommended_dtype_name,
    sae_decoder_direction,
    save_feature_handle,
    semantic_lure_cases,
    token_position_sweep_rows,
)

MODEL_ID = DEFAULT_MODEL_ID
SAE_REPO_ID = DEFAULT_QWEN_SCOPE_REPO_ID
DTYPE = recommended_dtype_name()
SAE_DEVICE = default_sae_device()
SAE_DTYPE = DTYPE
HANDLE_CACHE = root / "outputs" / "candidates" / "bat_ball_top_feature_answer_instruction.json"

lm = load_qwen_language_model(MODEL_ID, device_map="auto", dtype=DTYPE, dispatch=True)
print({"model": MODEL_ID, "sae_repo": SAE_REPO_ID, "dtype": DTYPE, "sae_device": SAE_DEVICE})


In [ ]:
CASE = instruct_lure_case(BAT_BALL_CASE)
LAYER = 14
TOP_N = 12

sae = load_qwen_scope_sae(
    SAE_REPO_ID,
    LAYER,
    device=SAE_DEVICE,
    dtype=dtype_from_name(SAE_DTYPE),
)
residual, _candidate_rows, candidates = candidate_feature_rows(lm, CASE, layer=LAYER, sae=sae, top_n=TOP_N)
_baseline, ranked = rank_lure_feature_effects(
    lm,
    CASE.prompt,
    correct_answer=CASE.correct_answer,
    lure_answer=CASE.lure_answer,
    layer=LAYER,
    sae=sae,
    residual=residual,
    candidate_features=candidates,
    top_n_candidates=TOP_N,
)
feature_ids = [row.feature_id for row in ranked[:8]]
rows = decoder_cosine_rows(sae, feature_ids)
display([r.as_row() for r in ranked[:8]])
display(rows)


해석 체크: 강한 후보끼리 cosine이 높으면 유사한 residual direction을 나눠 가진 feature family일 수 있습니다. cosine이 낮은데 효과가 비슷하면 서로 다른 경로로 같은 answer margin을 건드리는 후보입니다.